# Lesson 2.3 — 时间对齐（time alignment）：frame、timestamp 与 episode

本 notebook 回答三个决定一个 dataset 能否被用于训练的问题：

1. observation 与 action 是如何**配对**的？
2. 这里的 **timestamp** 究竟意味着什么？
3. 声明的 **frequency** 是否与实际情况相符？

第 2 个和第 3 个问题的答案与 metadata 所声称的不一致，本 notebook 通过演示而非断言来说明这一点。

前置条件：已经转换好的 dataset `datasets/lerobot/pickcube/`。第一个 cell 设置了一个 workspace 本地的 Hugging Face cache，因为默认的 cache 位置在此环境中不可写。

## 2.3.0 — 环境配置

In [1]:
import os
from pathlib import Path

from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
REPO_ROOT = PROJECT_ROOT

# LeRobot routes dataset parquet reads through the Hugging Face datasets cache.
# Point it inside the workspace so it stays writable and out of the home directory.
HF_CACHE = REPO_ROOT / ".cache" / "hf"
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_CACHE / "datasets"))

DATASET_ROOT = REPO_ROOT / "datasets" / "lerobot" / "pickcube"
print("repo root    :", REPO_ROOT)
print("dataset root :", DATASET_ROOT, "| exists:", DATASET_ROOT.exists())

repo root    : /home/bowenyuan/Projects/embodied-ai-learning
dataset root : /home/bowenyuan/Projects/embodied-ai-learning/datasets/lerobot/pickcube | exists: True


## 2.3.1 — Frame、timestamp 与 episode

- **frame** — 一个 timestep 条目：一个 observation、一个 action、一个 timestamp。
- **episode** — 一段连续的 rollout，从 reset 到 termination 或 truncation。
- **timestamp** — frame 被*记录*的时刻。所有与控制时序（control timing）相关的事情都取决于这个字段是否真实。

LeRobot 依据 dataset 声明的 FPS 把 `frame_index` 映射为秒。因此，声明的 FPS 静默地定义了建立在它之上的每一个时序模型的时间轴。

In [2]:
import numpy as np
import pyarrow.parquet as pq

from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(repo_id="pickcube", root=str(DATASET_ROOT))

print("number of frames  :", len(dataset))
print("number of episodes:", dataset.num_episodes)
print("declared FPS      :", dataset.fps)

sample = dataset[0]
print("\nsample keys:", sorted(sample.keys()))
print("observation.state:", tuple(sample["observation.state"].shape), sample["observation.state"].dtype)
print("action           :", tuple(sample["action"].shape), sample["action"].dtype)
print("timestamp        :", float(sample["timestamp"]), "| frame_index:", int(sample["frame_index"]))

number of frames  : 50
number of episodes: 1
declared FPS      : 50

sample keys: ['action', 'episode_index', 'frame_index', 'index', 'observation.state', 'task', 'task_index', 'timestamp']
observation.state: (42,) torch.float32
action           : (8,) torch.float32
timestamp        : 0.0 | frame_index: 0


## 2.3.2 — timestamp 究竟是什么

直接读取 parquet payload。如果 timestamp 是真实的采集时间，它们的间隔会有轻微变化。如果是合成的，间隔则会完全恒定。

这一区别并非学术性的：恒定的间隔意味着这些数字是由索引计算出来的，而不是测量得到的。

In [3]:
parquet_path = DATASET_ROOT / "data" / "chunk-000" / "file-000.parquet"
table = pq.read_table(parquet_path)
print("parquet schema:")
print(table.schema)

timestamps = table.column("timestamp").to_numpy()
frame_index = table.column("frame_index").to_numpy()
episode_index = table.column("episode_index").to_numpy()

print("\ntimestamp head:", np.round(timestamps[:5], 4))
print("timestamp tail:", np.round(timestamps[-3:], 4))
print("frame_index    :", frame_index[:5], "...", frame_index[-3:])
print("episode_index unique:", np.unique(episode_index))

intervals = np.diff(timestamps)
print("\nintervals (unique values):", np.unique(np.round(intervals, 6)))
print("all intervals identical   :", np.allclose(intervals, intervals[0]))

implied = np.arange(len(timestamps)) * intervals[0]
print("matches np.arange(T) * 0.02:", np.allclose(timestamps, implied, atol=1e-6))

parquet schema:
observation.state: fixed_size_list<element: float>[42]
  child 0, element: float
action: fixed_size_list<element: float>[8]
  child 0, element: float
timestamp: float
frame_index: int64
episode_index: int64
index: int64
task_index: int64
-- schema metadata --
huggingface: '{"info": {"features": {"observation.state": {"feature": {"d' + 458

timestamp head: [0.   0.02 0.04 0.06 0.08]
timestamp tail: [0.94 0.96 0.98]
frame_index    : [0 1 2 3 4] ... [47 48 49]
episode_index unique: [0]

intervals (unique values): [0.02]
all intervals identical   : True
matches np.arange(T) * 0.02: True


## 2.3.3 — 20 Hz / 50 Hz 缺陷

timestamp 是 `np.arange(T) * 0.02`，也就是合成的 50 Hz。仿真器（simulator）真实的控制频率是 **20 Hz**。把声明的值与环境自身的 `control_freq` 进行比较。

| 量 | 值 | 来源 |
|---|---:|---|
| dataset metadata 中声明的 `fps` | 50 | 由合成的 timestamp 推断而来 |
| 真实控制频率 | 20 | `env.unwrapped.control_freq` |
| 真实控制周期 | 0.05 s | `env.unwrapped.control_timestep` |

溯源链（provenance chain）是：

```text
collector stores no timing metadata
    -> timestamps synthesized as np.arange(T) * 0.02
        -> converter infers FPS from those timestamps
            -> info.json declares fps = 50 over 20 Hz data
```

**后果：** LeRobot 使用声明的 FPS 把 `frame_index` 转换为秒，因此时间轴被压缩了 **2.5 倍**。未来的时间窗口 `[B, T, D]` 覆盖的秒数将不是它表面上覆盖的秒数。

In [4]:
import gymnasium as gym
import mani_skill.envs

env = gym.make("PickCube-v1", obs_mode="state", control_mode="pd_joint_delta_pos", num_envs=1)
unwrapped = env.unwrapped

declared_fps = dataset.fps
real_control_freq = unwrapped.control_freq
real_control_period = unwrapped.control_timestep
sim_freq = unwrapped.sim_freq

print(f"declared FPS in dataset   : {declared_fps}")
print(f"real control_freq         : {real_control_freq} Hz")
print(f"real control_timestep     : {real_control_period} s")
print(f"sim_freq                  : {sim_freq} Hz  (physics steps per second)")
print(f"physics steps per action  : {int(sim_freq * real_control_period)}")
print()
print(f"declared period           : {1 / declared_fps:.4f} s")
print(f"real period               : {real_control_period:.4f} s")
print(f"time axis compression     : {real_control_period / (1 / declared_fps):.1f}x")

seconds_declared = len(dataset) / declared_fps
seconds_real = len(dataset) / real_control_freq
print(f"\n50 frames span {seconds_declared:.2f} s according to metadata, but {seconds_real:.2f} s of real control")
env.close()

2026-09-22 11:36:28,496 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


declared FPS in dataset   : 50
real control_freq         : 20 Hz
real control_timestep     : 0.05 s
sim_freq                  : 100 Hz  (physics steps per second)
physics steps per action  : 5

declared period           : 0.0200 s
real period               : 0.0500 s
time axis compression     : 2.5x

50 frames span 1.00 s according to metadata, but 2.50 s of real control


### 为什么这不能靠修改一个数字来修复

把 metadata 中的 `fps` 从 50 改成 20 会让声明的值变成真的，但底层的 timestamp 仍然是合成的。持久的修复要在源头进行：

- **collector** 必须记录它实际使用的控制频率，并且测量它的 timestamp 或明确标注其含义；
- **converter** 必须依据这个记录值来声明频率，而不是从它即将伪造出来的 timestamp 中推断频率。

在此之前，把该 dataset 视为 **20 Hz 控制下的 50 frames**，并把声明的 50 FPS 视为错误的。这一点已作为 open issue 记录在 `notes/progress.md` 中。

## 2.3.4 — 训练配对是 `(o_t, a_t)`，而不是 `(o_t, a_{t+1})`

配对方式是一种契约（contract），而不是一个习惯约定。如果 action 被偏移了一帧，那么每个 target 都是错的，而所有 shape 却保持完全相同——模型会顺利地训练下去，并学会预测下一个 action。

在存储的数据上验证这条规则：索引 `t` 处的 action 就是施加到索引 `t` 处 observation 上的 action。

In [5]:
# The stored arrays are aligned by construction, so demonstrate the contrast
# between the correct pairing and the off-by-one pairing.

actions = table.column("action").to_pylist()
states = table.column("observation.state").to_pylist()

T = len(actions)
print("T actions :", T)
print("T states  :", len(states))

correct_pairs = [(t, t) for t in range(T)]
shifted_pairs = [(t, t + 1) for t in range(T - 1)]

print("\ncorrect pairing  : (o_t, a_t)     ->", correct_pairs[:3], "...")
print("off-by-one pairing: (o_t, a_t+1)   ->", shifted_pairs[:3], "...")
print("\nboth produce the same tensor shapes;")
print("only the first matches the environment transition o_t -a_t-> o_{t+1}.")

T actions : 50
T states  : 50

correct pairing  : (o_t, a_t)     -> [(0, 0), (1, 1), (2, 2)] ...
off-by-one pairing: (o_t, a_t+1)   -> [(0, 1), (1, 2), (2, 3)] ...

both produce the same tensor shapes;
only the first matches the environment transition o_t -a_t-> o_{t+1}.


### `T` 与 `T+1` 的问题

一个包含 T 个 action 的 episode 涉及 **T+1** 个 state：

```text
o_0 --a_0--> o_1 --a_1--> o_2 ... o_{T-1} --a_{T-1}--> o_T
```

存储的 observation 数组保存的是 `o_0 .. o_{T-1}`——即 policy *看到*的那些 state。终止 observation `o_T`（以及原始 HDF5 中的 `next_observations`）是一个单独的字段。把两者混为一谈，会得到一个看起来完整、却静默地缺失了最后一次 transition 的 dataset。

In [6]:
import h5py

h5_path = REPO_ROOT / "datasets" / "pickcube" / "random_episode_standard.h5"
with h5py.File(h5_path, "r") as handle:
    print("standardized HDF5 keys and shapes:")
    for key in handle:
        if key == "metadata":
            print(f"  {key:<18} (group) attrs={dict(handle[key].attrs)}")
        else:
            print(f"  {key:<18} shape={handle[key].shape} dtype={handle[key].dtype}")

print("\nNote timestamps were stored as a separate array:")
with h5py.File(h5_path, "r") as handle:
    stamps = handle["timestamps"][:]
print("  first 5:", stamps[:5])
print("  step    :", np.unique(np.round(np.diff(stamps), 6)))

standardized HDF5 keys and shapes:
  actions            shape=(50, 8) dtype=float32
  metadata           (group) attrs={'control_mode': 'pd_joint_delta_pos', 'robot': 'Panda', 'source': 'ManiSkill', 'task': 'PickCube-v1'}
  observations       shape=(50, 42) dtype=float32
  rewards            shape=(50, 1) dtype=float32
  timestamps         shape=(50,) dtype=float64

Note timestamps were stored as a separate array:
  first 5: [0.   0.02 0.04 0.06 0.08]
  step    : [0.02]


## 小结

1. 该 dataset 的 timestamp 是合成的（`np.arange(T) * 0.02`），而不是测量得到的——间隔完全恒定，这就是破绽所在。
2. 声明的 50 FPS 与环境真实的 **20 Hz** 控制频率相矛盾，把时间轴压缩了 2.5 倍。不要静默地修改 `fps`；要修复 collector 和 converter。
3. 训练配对是 `(o_t, a_t)`。发生 off-by-one 的配对具有完全相同的 shape，但它是错的。
4. 一个包含 T 个 action 的 episode 有 T+1 个 state。存储的 observation 数组有 T 个。

下一步：`2.4_observation_schema.ipynb` 讲解 observation/action schema，然后是 `2.5_generate_lerobot_dataset.ipynb` 讲解转换。